# FloodEye: RQ2 trên Kaggle T4

Notebook chỉ lấy mã ở một commit cố định, chuẩn bị dữ liệu bằng script của kho, rồi gọi `scripts/run_rq2.py`: xuất E2c sang TensorRT FP16 và INT8, chấm độ chính xác trên SeaDronesSee và VisDrone test-dev, đo thời gian cả pipeline trên một đoạn video 4K có telemetry thật, ghép thành bảng recall theo FPS. Mọi logic nằm trong kho mã.

Trước khi chạy: bật **GPU T4** và **Internet** ở khung Settings (notebook cài TensorRT 10; ultralytics tự cài `onnx` khi xuất).

Thêm qua **Add Input** (khung bên phải):
- dataset chứa `best.pt` của E2c (tải `runs/train/E2c_s0/weights/best.pt` lên thành một Kaggle Dataset);
- AFO (`jangsienicajzkowy/afo-aerial-dataset-of-floating-objects`): ảnh val để hiệu chỉnh INT8;
- SeaDronesSee-MOT (`rameezakther/seadronessee-mot-sampled-dataset`): tập đánh giá và đoạn video đo tốc độ.

Cách dùng: lần đầu đặt `ONLY = ["pt_tiled", "fp16_tiled"]`, **Run All** trong phiên tương tác để kiểm tra (khoảng 20 phút). Ổn thì đặt `ONLY = None`, **Save Version → Save & Run All (Commit)**. Bước nào đã có kết quả thì bỏ qua, nên phiên bị ngắt thì Add Input phần Output cũ và chạy lại.

Số đo trên T4 chỉ để so sánh tương đối giữa các cấu hình, không quy đổi ra FPS trên Jetson (TECHNICAL mục 14.4).

In [ ]:
COMMIT = "d6750853b615bd36c62d16c51b2a0c000a06434d"  # mã commit đầy đủ của kho
CONFIG = "configs/experiments/rq2_t4.yaml"
ONLY = ["pt_tiled", "fp16_tiled"]  # None: mọi biến thể trong CONFIG
STEPS = None  # None: export accuracy speed tradeoff; hoặc một phần, ví dụ ["accuracy", "tradeoff"]

In [ ]:
import os

if not os.path.isdir("/kaggle/working/floodeye"):
    !git clone -q https://github.com/huzy179/uav-rescue-qa.git /kaggle/working/floodeye
%cd /kaggle/working/floodeye
!git fetch -q && git checkout -q {COMMIT} && git log -1 --oneline
!pip install -q -e ".[dev]"
# Ghim TensorRT 10: với TensorRT 11 (pip mặc định), ultralytics xuất INT8 dạng Q/DQ qua ModelOpt và build lỗi trên T4 (7/11)
# ("Could not find any implementation" ở Conv INT8). Bản 10 dùng hiệu chỉnh INT8 cổ điển, chạy được trên T4.
import torch
!pip install -q "tensorrt-cu{torch.version.cuda.split('.')[0]}==10.16.1.11"


In [ ]:
import torch, cv2, ultralytics, tensorrt

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("opencv", cv2.__version__, "| ultralytics", ultralytics.__version__)
assert torch.cuda.is_available(), "Chưa có GPU: bật Accelerator GPU T4 ở khung Settings"
print("tensorrt", tensorrt.__version__)
assert tensorrt.__version__.startswith("10."), "Cần TensorRT 10 (INT8 của TensorRT 11 không chạy trên T4): chạy lại ô cài đặt"


## Trọng số và dữ liệu

`best.pt` của E2c chép về đúng đường dẫn trong `CONFIG`. Phiên trước đã có Output (`runs/rq2/`) thì chép lại để bỏ qua bước đã xong.

In [ ]:
import glob, shutil, yaml

cfg = yaml.safe_load(open(CONFIG))
hits = [p for p in glob.glob("/kaggle/input/**/best.pt", recursive=True) if "E3" not in p]
if len(hits) != 1:
    raise RuntimeError(f"Cần đúng một best.pt của E2c trong /kaggle/input, tìm thấy: {hits}")
os.makedirs(os.path.dirname(cfg["weights"]), exist_ok=True)
shutil.copy(hits[0], cfg["weights"])
!sha256sum {cfg["weights"]}

old = glob.glob("/kaggle/input/**/rq2/rq2_config.yaml", recursive=True)
if old:
    print("Chép kết quả phiên trước:", os.path.dirname(old[0]))
    shutil.copytree(os.path.dirname(old[0]), cfg["output"], dirs_exist_ok=True)


In [ ]:
%%bash
set -e
mkdir -p data/raw/visdrone
for split in val test-dev; do
  if [ ! -d data/raw/visdrone/VisDrone2019-DET-$split ]; then
    (cd data/raw/visdrone && curl -sLO https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-$split.zip \
      && unzip -q VisDrone2019-DET-$split.zip && rm VisDrone2019-DET-$split.zip)
  fi
  python tools/convert_visdrone.py --src data/raw/visdrone/VisDrone2019-DET-$split --split $split
done

In [ ]:
from pathlib import Path

hits = sorted(Path("/kaggle/input").glob("**/6categories.names"))
if len(hits) != 1:
    raise RuntimeError(f"Cần đúng một bản AFO (Add Input) để hiệu chỉnh INT8, tìm thấy: {hits}")
afo_root = hits[0].parent.parent.parent
!python tools/prepare_afo.py --src {afo_root} --out data/raw/afo_yolo --max-side 1920
!python tools/convert_yolo_subset.py --src data/raw/afo_yolo --dst data/processed/afo --names-file data/raw/afo_yolo/classes.txt --keep human --splits val

In [ ]:
hits = sorted({p.parent.parent for p in Path("/kaggle/input").glob("**/annotations/val.json")})
if len(hits) != 1:
    raise RuntimeError(f"Cần đúng một bản SeaDronesSee-MOT (Add Input), tìm thấy: {hits}")
sds = hits[0]
!python tools/make_sds_det.py --src {sds} --dst data/processed/sds_det --split val --every 15
!python tools/convert_seadronessee_mot.py --src {sds} --dst data/processed/sds_mot --splits val

## Chạy RQ2

Xuất engine mất vài phút mỗi cái (INT8 lâu hơn vì hiệu chỉnh). Log có `[bỏ qua]` cho biến thể thu nhỏ theo độ cao trên VisDrone: tập đó không có độ cao, đúng như thiết kế.

In [ ]:
only = f"--only {' '.join(ONLY)}" if ONLY else ""
steps = f"--steps {' '.join(STEPS)}" if STEPS else ""
!CUDA_VISIBLE_DEVICES=0 python scripts/run_rq2.py --config {CONFIG} --device 0 {only} {steps}

In [ ]:
import pandas as pd

print(pd.read_csv(f"{cfg['output']}/rq2.csv").to_string(index=False))

## Dọn dẹp

Xoá dữ liệu và trọng số để Output chỉ còn `runs/rq2/` (bảng, hình, log, `run_config.json`, engine). Engine chỉ chạy được trên đúng loại GPU và bản TensorRT đã xuất, nên giữ lại để chạy tiếp phiên sau. Tải `runs/rq2/` về máy, bỏ thư mục `models/` nếu không cần.

In [ ]:
!rm -rf data runs/train
!find runs/rq2 -name "*.pt" -delete